# Train model_behavior (Google Colab)
Notebook MVP de tao fake behavior data, train Transformer nho va export profile cho RAG service.

In [ ]:
!pip install -q torch pandas scikit-learn

In [ ]:
import json
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.cluster import KMeans
from torch.utils.data import DataLoader, Dataset

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

event_vocab = {'view': 1, 'search': 2, 'cart': 3, 'purchase': 4, 'chat': 5}
category_vocab = {'gaming': 1, 'ultrabook': 2, 'workstation': 3, 'budget': 4, 'clothes': 5}
item_vocab_size = 200

def sample_user_events(user_idx: int, seq_len: int = 20):
    segment_type = random.choice(['tech_enthusiast', 'bargain_hunter', 'brand_loyalist', 'new_user'])
    events = []
    for _ in range(seq_len):
        if segment_type == 'tech_enthusiast':
            category = random.choice(['gaming', 'workstation', 'ultrabook'])
            event = random.choice(['view', 'search', 'chat', 'cart'])
        elif segment_type == 'bargain_hunter':
            category = random.choice(['budget', 'clothes'])
            event = random.choice(['search', 'view', 'cart'])
        elif segment_type == 'brand_loyalist':
            category = random.choice(['ultrabook', 'workstation'])
            event = random.choice(['view', 'search', 'purchase'])
        else:
            category = random.choice(list(category_vocab.keys()))
            event = random.choice(list(event_vocab.keys()))

        item_id = random.randint(1, item_vocab_size - 1)
        events.append({
            'user_id': f'US_{user_idx:03d}',
            'segment_hint': segment_type,
            'event_type': event,
            'category': category,
            'item_id': item_id
        })
    return events

rows = []
for i in range(1, 301):
    rows.extend(sample_user_events(i, seq_len=20))

df = pd.DataFrame(rows)
df.head()

In [ ]:
class BehaviorSeqDataset(Dataset):
    def __init__(self, frame, seq_len=10):
        self.seq_len = seq_len
        self.samples = []
        for user_id, group in frame.groupby('user_id'):
            item_seq = group['item_id'].tolist()
            cat_seq = [category_vocab[c] for c in group['category'].tolist()]
            evt_seq = [event_vocab[e] for e in group['event_type'].tolist()]
            for i in range(len(item_seq) - seq_len):
                self.samples.append((
                    item_seq[i:i+seq_len],
                    cat_seq[i:i+seq_len],
                    evt_seq[i:i+seq_len],
                    item_seq[i+seq_len]
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item_seq, cat_seq, evt_seq, target = self.samples[idx]
        pos = list(range(len(item_seq)))
        return (
            torch.tensor(item_seq, dtype=torch.long),
            torch.tensor(cat_seq, dtype=torch.long),
            torch.tensor(evt_seq, dtype=torch.long),
            torch.tensor(pos, dtype=torch.long),
            torch.tensor(target, dtype=torch.long),
        )

In [ ]:
class BehaviorTransformer(nn.Module):
    def __init__(self, num_items, num_categories, embed_dim=128, num_heads=8, num_layers=2):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items + 1, embed_dim, padding_idx=0)
        self.cat_embedding = nn.Embedding(num_categories + 1, embed_dim)
        self.event_embedding = nn.Embedding(6, embed_dim)
        self.positional_enc = nn.Embedding(64, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=256, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(embed_dim, num_items)

    def forward(self, item_seq, cat_seq, event_seq, positions):
        x = self.item_embedding(item_seq)
        x = x + 0.2 * self.cat_embedding(cat_seq) + 0.2 * self.event_embedding(event_seq)
        x = x + self.positional_enc(positions)
        x = self.transformer(x)
        return self.output_layer(x[:, -1, :])

In [ ]:
dataset = BehaviorSeqDataset(df, seq_len=10)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = BehaviorTransformer(num_items=item_vocab_size, num_categories=len(category_vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    total_loss = 0.0
    for item_seq, cat_seq, evt_seq, pos, target in loader:
        item_seq, cat_seq, evt_seq, pos, target = [x.to(device) for x in (item_seq, cat_seq, evt_seq, pos, target)]
        logits = model(item_seq, cat_seq, evt_seq, pos)
        loss = criterion(logits, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}: loss={total_loss/len(loader):.4f}')

In [ ]:
# Tao user embedding don gian va segment bang KMeans cho MVP
user_vectors = []
user_ids = []
for uid, group in df.groupby('user_id'):
    vec = [
        group['item_id'].mean(),
        group['item_id'].std(),
        (group['event_type'] == 'search').mean(),
        (group['event_type'] == 'cart').mean(),
        (group['event_type'] == 'purchase').mean(),
    ]
    user_vectors.append(np.nan_to_num(vec).tolist())
    user_ids.append(uid)

kmeans = KMeans(n_clusters=6, random_state=SEED, n_init='auto')
labels = kmeans.fit_predict(np.array(user_vectors))

segment_names = {
    0: 'high_value_buyer',
    1: 'bargain_hunter',
    2: 'brand_loyalist',
    3: 'tech_enthusiast',
    4: 'occasional_shopper',
    5: 'new_user'
}

payload = {'users': []}
for uid, label in zip(user_ids, labels):
    sub = df[df['user_id'] == uid]
    viewed_categories = sub['category'].value_counts().head(2).index.tolist()
    payload['users'].append({
        'user_id': uid,
        'segment': segment_names.get(int(label), 'new_user'),
        'viewed_categories': viewed_categories,
        'price_sensitivity': random.choice(['low', 'medium', 'high']),
        'brand_preference': random.sample(['Apple', 'ASUS', 'Dell', 'Lenovo', 'MSI', 'Acer'], 2),
        'session_duration': int(random.randint(180, 2200))
    })

with open('fake_user_behavior.json', 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

torch.save(model.state_dict(), 'behavior_transformer.pt')
print('Saved: fake_user_behavior.json and behavior_transformer.pt')